In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dipankarsrirag/topic-modelling-on-emails")

print("Path to dataset files:", path)

100%|██████████| 10.4M/10.4M [00:00<00:00, 103MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dipankarsrirag/topic-modelling-on-emails/versions/1


In [ ]:
import shutil

# ضغط المجلد
shutil.make_archive('dataset', 'zip', path)

# تحميل الملف المضغوط
from google.colab import files
files.download('dataset.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import os

files = os.listdir(path)
print(files)

['Data']


In [3]:
data_path = path + "/Data"

files = os.listdir(data_path)
print(files)

['Science', 'Entertainment', 'Crime', 'Politics']


In [4]:
import os
import pandas as pd

texts = []
labels = []

for label in os.listdir(data_path):
    folder_path = os.path.join(data_path, label)

    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)

            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    texts.append(f.read())
                    labels.append(label)
            except:
                continue

df = pd.DataFrame({
    'text': texts,
    'label': labels
})

print(df['label'].value_counts())

label
Science          4000
Politics         3001
Crime            1100
Entertainment    1053
Name: count, dtype: int64


In [5]:
print("عدد البيانات:", len(df))

عدد البيانات: 9154


In [6]:
df.dropna(inplace=True)

In [7]:
print("عدد البيانات:", len(df))

عدد البيانات: 9154


In [8]:
df['label'].value_counts()

,count
label,
Science,4000
Politics,3001
Crime,1100
Entertainment,1053


In [9]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [10]:
print (df.head())

                                                text    label
0  in article chcbo joy zoo toronto edu henry zoo...  Science
1  in article apr eff org a charles gross acg eff...  Science
2  distribution usa message id qvjvr dms opus dgi...  Science
3  does anyone know of a non word password genera...  Science
4  distribution world message id rq uj news inter...  Science


In [14]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================

# =========================
X = df['text']
y = df['label']

# =========================

# =========================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM (LinearSVC)": LinearSVC(),
    "Naive Bayes": MultinomialNB()
}

#
f1_results = {}

# =========================
#
# =========================
for name, model in models.items():

    print("=" * 70)
    print(f"MODEL: {name}")

    pipeline = ImbPipeline([
        ('tfidf', TfidfVectorizer(
            max_features=15000,
            ngram_range=(1,2),
            min_df=2,
            max_df=0.9,stop_words='english'
        )),
        ('ros', RandomOverSampler(random_state=42)),
        ('model', model)
    ])

    #
    y_pred = cross_val_predict(
        pipeline,
        X,
        y,
        cv=5
    )




    print(classification_report(y, y_pred))

  #
    f1 = f1_score(y, y_pred, average='weighted')
    f1_results[name] = f1

# =========================
#
# =========================
best_model = max(f1_results, key=f1_results.get)

print("\n" + "=" * 70)
print("🏆 BEST MODEL SELECTED")
print("Model:", best_model)
print("Weighted F1-score:", f1_results[best_model])

MODEL: Logistic Regression
               precision    recall  f1-score   support

        Crime       0.32      0.34      0.33      1100
Entertainment       0.33      0.45      0.38      1053
     Politics       0.97      0.96      0.96      3001
      Science       0.86      0.77      0.81      4000

     accuracy                           0.74      9154
    macro avg       0.62      0.63      0.62      9154
 weighted avg       0.77      0.74      0.75      9154

MODEL: SVM (LinearSVC)
               precision    recall  f1-score   support

        Crime       0.32      0.32      0.32      1100
Entertainment       0.32      0.40      0.36      1053
     Politics       0.93      0.97      0.95      3001
      Science       0.85      0.77      0.80      4000

     accuracy                           0.74      9154
    macro avg       0.60      0.61      0.61      9154
 weighted avg       0.75      0.74      0.74      9154

MODEL: Naive Bayes
               precision    recall  f1-score 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# =========================
# البيانات
# =========================
X = df['text']
y = df['label']

# =========================
# النماذج
# =========================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM (LinearSVC)": LinearSVC(),
    "Naive Bayes": MultinomialNB()
}

# =========================
# تقييم لكل نموذج على كل فئة
# =========================
for name, model in models.items():

    print("=" * 70)
    print(f"MODEL: {name}")

    pipeline = ImbPipeline([
        ('tfidf', TfidfVectorizer(
            max_features=15000,
            ngram_range=(1,2),
            min_df=2,
            max_df=0.9
        )),
        ('ros', RandomOverSampler(random_state=42)),
        ('model', model)
    ])

    # تنبؤ Cross Validation (بدون تسريب بيانات)
    y_pred = cross_val_predict(
        pipeline,
        X,
        y,
        cv=5
    )

    # تقرير مفصل لكل فئة
    report = classification_report(y, y_pred)

    print(report)

MODEL: Logistic Regression
               precision    recall  f1-score   support

        Crime       0.33      0.36      0.34      1100
Entertainment       0.33      0.43      0.37      1053
     Politics       0.95      0.96      0.96      3001
      Science       0.86      0.76      0.81      4000

     accuracy                           0.74      9154
    macro avg       0.62      0.63      0.62      9154
 weighted avg       0.76      0.74      0.75      9154

MODEL: SVM (LinearSVC)
               precision    recall  f1-score   support

        Crime       0.33      0.35      0.34      1100
Entertainment       0.33      0.41      0.36      1053
     Politics       0.93      0.97      0.95      3001
      Science       0.86      0.76      0.81      4000

     accuracy                           0.74      9154
    macro avg       0.61      0.62      0.61      9154
 weighted avg       0.76      0.74      0.75      9154

MODEL: Naive Bayes
               precision    recall  f1-score 

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from imblearn.over_sampling import RandomOverSampler

# =========================
# البيانات
# =========================
X = df['text']
y = df['label']

# =========================
# التقسيم أولاً (صح)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# تحويل النصوص إلى TF-IDF
# =========================
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,1),
    min_df=2,
    max_df=0.9
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# =========================
# موازنة البيانات (بعد TF-IDF)
# =========================
ros = RandomOverSampler(random_state=42)

X_train_balanced, y_train_balanced = ros.fit_resample(
    X_train_tfidf,
    y_train
)

# =========================
# النماذج
# =========================
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,

    ),
    "SVM": LinearSVC(

    ),
    "Naive Bayes": MultinomialNB()
}

# =========================
# التدريب والتقييم
# =========================
results = {}

for name, model in models.items():
    print("=" * 50)
    print(f"Training {name}")

    model.fit(X_train_balanced, y_train_balanced)

    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"{name} Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))


# =========================
# أفضل نموذج
# =========================
print("\n" + "=" * 50)
print("Final Results")

for model_name, score in results.items():
    print(f"{model_name}: {score:.4f}")

best_model = max(results, key=results.get)
print(f"\nBest Model: {best_model}")
print(f"Best Accuracy: {results[best_model]:.4f}")

Training Logistic Regression
Logistic Regression Accuracy: 0.6543
               precision    recall  f1-score   support

        Crime       0.08      0.11      0.09       220
Entertainment       0.08      0.10      0.09       211
     Politics       0.98      0.96      0.97       600
      Science       0.90      0.72      0.80       800

     accuracy                           0.65      1831
    macro avg       0.51      0.47      0.49      1831
 weighted avg       0.73      0.65      0.69      1831

Training SVM
SVM Accuracy: 0.6412
               precision    recall  f1-score   support

        Crime       0.02      0.03      0.02       220
Entertainment       0.03      0.04      0.04       211
     Politics       0.98      0.98      0.98       600
      Science       0.86      0.71      0.78       800

     accuracy                           0.64      1831
    macro avg       0.47      0.44      0.45      1831
 weighted avg       0.70      0.64      0.67      1831

Training Naive

In [ ]:
print(df['label'].value_counts())

label
Science          4000
Politics         3001
Crime            1100
Entertainment    1053
Name: count, dtype: int64


In [ ]:
for label in os.listdir(data_path):
    folder = os.path.join(data_path, label)
    print(label, "عدد الملفات:", len(os.listdir(folder)))

Entertainment عدد الملفات: 1053
Crime عدد الملفات: 1100
Politics عدد الملفات: 3001
Science عدد الملفات: 4001
